# Configurable GPS Acquisition + Tracking (L1CA or L2C)

This notebook demonstrates a unified acquisition/tracking flow built on `SignalDefinition` from `utils.signal_interfaces`.

A signal is described by data rather than by a bespoke correlator: `utils.code_components` says how each spreading code occupies the chip clock, and a `LoopDiscriminatorPolicy` says which component drives which loop. One `utils.tracking_channel.TrackingChannel` then serves every signal.

| Signal | Components | How they are multiplexed | Carrier loop runs on |
|---|---|---|---|
| GPS L1 C/A | `CA` | single code, 1.023 Mcps | `CA` |
| GPS L2C | `CM`, `CL` | interleaved on alternating chip slots, 1.023 Mcps combined | `CM` |

Set `SIGNAL_FAMILY` in the config cell to switch.



In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming, tracking_channel
from utils.signal_interfaces import (
    SignalFamily,
    build_signal_definitions,
    build_acquisition_code_params,
    create_tracking_channels,
)

plt.rcParams.update({"font.size": 14})

In [ ]:
# ----- User configuration -----
SIGNAL_FAMILY = SignalFamily.L2C  # SignalFamily.L1CA or SignalFamily.L2C
EXPERIMENT_INDEX = 2

# Acquisition settings per family.
#
# `replica_duration_ms` is the coherent integration length, and it sets the
# Doppler bin width to 1 / T_coherent.  L1 C/A can integrate over several code
# periods; L2C acquires on CM alone, whose period is 20 ms.
#
ACQ_SETTINGS = {
    SignalFamily.L1CA: dict(replica_duration_ms=4, num_blocks=8, p_fa_total=1e-9, half_bin=False),
    SignalFamily.L2C: dict(replica_duration_ms=20, num_blocks=1, p_fa_total=1e-6, half_bin=False),
}

# Band id to look for in the collect metadata.
TARGET_BAND = {
    SignalFamily.L1CA: "L1",
    SignalFamily.L2C: "L2",
}

# Tracking settings
BUFFER_DURATION_MS = 200
BLOCK_DURATION_MS = 1
TRACK_DURATION_MS = 2000
START_TRACKING_IN_PLL_MODE = False


print(f"Signal family: {SIGNAL_FAMILY.value}")

In [ ]:
local_data_dir = Path(utils.__file__).parent.parent / "local-data"
collects_dir = local_data_dir / "collects"
available_experiment_names = sorted(fp.name for fp in collects_dir.iterdir())
print("Available experiments:", ", ".join(available_experiment_names))

experiment_name = available_experiment_names[EXPERIMENT_INDEX]
experiment_dir = collects_dir / experiment_name
print(f"Selected experiment: {experiment_name}")

metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

target_band = TARGET_BAND[SIGNAL_FAMILY]
candidate_collect_ids = []
for cid in metadata.collect_ids:
    collect_cfg = metadata.collects[cid]
    channel_cfg = metadata.channel_configurations[collect_cfg.channel_config_id]
    if target_band in channel_cfg.band_ids:
        candidate_collect_ids.append(cid)

if len(candidate_collect_ids) == 0:
    raise RuntimeError(
        f"No collect found for target band {target_band!r}. "
        f"Bands present in this experiment: {sorted(metadata.band_ids)}"
    )

collect_id = candidate_collect_ids[0]
band_id = target_band
collect_config = metadata.collects[collect_id]
channel_config = metadata.channel_configurations[collect_config.channel_config_id]
band_config = metadata.band_configurations[band_id]

samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
inter_freq_hz = band_config.inter_freq
collect_filepath = experiment_dir / collect_config.filename

print(f"Selected collect_id={collect_id}, band_id={band_id}")
print(f"Collect filepath: {collect_filepath}")
print(f"Sample rate: {samp_rate} Hz")

# The correlator steps the chip index once per sample, so anything below about
# two samples per chip silently skips chips rather than failing loudly.
_chip_rate = build_signal_definitions(SIGNAL_FAMILY, prns=[1])["G01"].tracking_code_rate_chips_per_sec
_samples_per_chip = samp_rate / _chip_rate
print(f"Chip rate: {_chip_rate/1e6:.3f} Mcps  ->  {_samples_per_chip:.2f} samples/chip")
if _samples_per_chip < 2.0:
    raise RuntimeError(
        f"Sample rate {samp_rate/1e6:.1f} Msps gives only {_samples_per_chip:.2f} samples per "
        f"chip at {_chip_rate/1e6:.3f} Mcps. The correlator needs at least ~2; "
        "this collect cannot be used for this signal without resampling."
    )

In [ ]:
signal_definitions = build_signal_definitions(SIGNAL_FAMILY)
acq_code_params = build_acquisition_code_params(signal_definitions)

acq_settings = ACQ_SETTINGS[SIGNAL_FAMILY]

# Read enough samples for the full coherent x non-coherent acquisition dwell.
acq_buffer_duration_ms = acq_settings["replica_duration_ms"] * acq_settings["num_blocks"]
acq_buffer_size_samples = int(samp_rate * acq_buffer_duration_ms / 1e3)
byte_buffer = bytearray(
    sample_streaming.compute_sample_array_size_bytes(
        acq_buffer_size_samples, sample_params.bit_depth, sample_params.is_complex
    )
)
samples = np.zeros(acq_buffer_size_samples, dtype=np.complex64)
baseband_samples = np.zeros(acq_buffer_size_samples, dtype=np.complex64)

with open(collect_filepath, "rb") as f:
    f.readinto(byte_buffer)

sample_streaming.convert_to_complex64_samples(byte_buffer, samples, sample_params)
sample_streaming.mixdown_samples(
    samples,
    baseband_samples,
    samp_rate,
    initial_phase_cycles=0.0,
    freq_hz=inter_freq_hz,
)
baseband_samples -= np.mean(baseband_samples)

acq_config = bpsk_acquisition.AcquisitionConfiguration(
    replica_duration_ms=acq_settings["replica_duration_ms"],
    num_blocks=acq_settings["num_blocks"],
    sample_rate=samp_rate,
    min_search_doppler_hz=-5000,
    max_search_doppler_hz=5000,
)
num_detection_tests = acq_config.num_doppler_bins * acq_config.replica_length_samples
p_fa_bin = 1 - (1 - acq_settings["p_fa_total"]) ** (1 / num_detection_tests)

doppler_bin_hz = acq_config.fft_resolution
worst_case_error_hz = doppler_bin_hz / (4 if acq_settings["half_bin"] else 2)
print(f"Doppler bin width: {doppler_bin_hz:.1f} Hz "
      f"(half-bin search {'on' if acq_settings['half_bin'] else 'off'}) "
      f"-> worst-case seeding error {worst_case_error_hz:.1f} Hz")

acq_results = bpsk_acquisition.run_acquisition(
    sample_block=baseband_samples,
    sample_block_uptime_epoch_ms=0.0,
    acq_config=acq_config,
    code_parameters=acq_code_params,
    prob_false_alaram=p_fa_bin,
    print_progress=True,
    noise_var_method="abscorrvar",
    half_bin_doppler_search=acq_settings["half_bin"],
)

acquired_signal_ids = sorted([
    sid for sid, result in acq_results.items() if result.signal_detected
])
print(f"Acquired signals ({SIGNAL_FAMILY.value}): {', '.join(acquired_signal_ids)}")

In [ ]:
if len(acquired_signal_ids) == 0:
    raise RuntimeError("No signals acquired; cannot start tracking.")

buffer_size_samples = int(samp_rate * BUFFER_DURATION_MS / 1e3)
num_buffers_to_process = TRACK_DURATION_MS // BUFFER_DURATION_MS
output_capacity = TRACK_DURATION_MS // BLOCK_DURATION_MS

tracking_loop_params = tracking_channel.TrackingLoopParameters(
    DLL_bandwidth_hz=2.0,
    PLL_bandwidth_hz=20.0,
    FLL_bandwidth_hz=50.0,
    nominal_update_period_ms=BLOCK_DURATION_MS,
    corr_period_ms=BLOCK_DURATION_MS,
    EPL_chip_spacing=0.5,
    prompt_corr_circ_length_threshold=0.9,
)

tracking_channels = create_tracking_channels(
    signal_definitions=signal_definitions,
    acquisition_results=acq_results,
    tracking_signal_ids=acquired_signal_ids,
    loop_params=tracking_loop_params,
    output_capacity=output_capacity,
    start_mode_pll=START_TRACKING_IN_PLL_MODE,
)

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
) as sample_stream:
    sample_buffer_generator = sample_stream.sample_buffer_generator()
    for i_buffer, buffer_samples in enumerate(sample_buffer_generator):
        if i_buffer >= num_buffers_to_process:
            break
        uptime_ms = i_buffer * BUFFER_DURATION_MS
        mixdown_phase_cycles = inter_freq_hz * (uptime_ms * 1e-3)
        sample_streaming.mixdown_samples(
            buffer_samples,
            buffer_samples,
            samp_rate,
            initial_phase_cycles=mixdown_phase_cycles,
            freq_hz=inter_freq_hz,
        )

        sample_buffer = sample_streaming.SampleBuffer(
            samples=buffer_samples,
            start_uptime_ms=uptime_ms,
            samp_rate=samp_rate,
        )
        for adapter in tracking_channels.values():
            adapter.process_sample_buffer(sample_buffer)

print("Tracking complete")

In [ ]:
plot_sig_id = acquired_signal_ids[0]
adapter = tracking_channels[plot_sig_id]
outputs = adapter.outputs
component_names = adapter.signal_definition.component_names

# The component the carrier loop runs on: CA for L1 C/A, CM for L2C, the Q pilot
# for L5.  That is the one whose I/Q constellation should be collapsed onto the
# real axis by the PLL.
carrier_index = adapter.signal_definition.discriminator_policy.carrier_component
carrier_prompt = adapter.get_prompt_component(component=carrier_index)

plot_time = outputs.uptime_epoch_ms * 1e-3

fig = plt.figure(figsize=(12, 8), dpi=150)
axes = fig.subplots(2, 1, sharex=True)

axes[0].scatter(plot_time, carrier_prompt.real, s=2, color="tab:red", label="In-phase")
axes[0].scatter(plot_time, carrier_prompt.imag, s=2, color="tab:blue", label="Quadrature")
axes[0].set_ylabel(f"Prompt ({component_names[carrier_index]})")
axes[0].set_title(f"{SIGNAL_FAMILY.value} tracking: {plot_sig_id}")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(plot_time, outputs.doppler_freq_hz, lw=1.5, color="tab:green")
axes[1].set_ylabel("Doppler [Hz]")
axes[1].set_xlabel("Uptime [s]")
axes[1].grid(True)

# Multi-component signals: compare the components' prompt magnitudes.
#   L2C  - CM and CL each occupy half the chip slots, so both sit near half the
#          magnitude a single full-rate code would reach.
#   L5   - I and Q are equal power and should track each other closely.
if len(component_names) > 1:
    fig2 = plt.figure(figsize=(12, 4), dpi=150)
    ax2 = fig2.add_subplot(1, 1, 1)
    for index, (name, color) in enumerate(zip(component_names, ("tab:purple", "tab:orange"))):
        prompt = adapter.get_prompt_component(component=index)
        ax2.scatter(plot_time, np.abs(prompt), s=2, label=f"{name} |Prompt|", color=color)
    ax2.set_title(f"{SIGNAL_FAMILY.value} components: {plot_sig_id}")
    ax2.set_ylabel("Magnitude")
    ax2.set_xlabel("Uptime [s]")
    ax2.grid(True)
    ax2.legend()

plt.show()